# STD_LR - Standard Logistic Regression

This notebook:
- Loads preprocessed data from BASE.ipynb
- Trains a Standard Logistic Regression model
- Evaluates and saves results to JSON (no graphs)

In [1]:
# Import Libraries
import time
import pickle
import json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 1. Load Preprocessed Data

In [2]:
# Load data from the output of data_preprocessor.ipynb
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

# DP norm precondition: clip every row to unit L2 norm (||x_i||_2 <= 1).
X_train = X_train / np.maximum(np.linalg.norm(X_train, axis=1, keepdims=True), 1.0)
X_test = X_test / np.maximum(np.linalg.norm(X_test, axis=1, keepdims=True), 1.0)
data_norm = 1.0

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set: {X_test.shape[0]} samples")

Training Set: 455 samples
Testing Set: 114 samples


## 2. Train Standard Logistic Regression

In [3]:
# Initialize and train the model
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

_t0 = time.perf_counter()
model.fit(X_train, y_train)
train_time = time.perf_counter() - _t0
# Score the non-private fit on the training partition too, so the
# generalisation gap can be read next to the held-out metrics.
y_train_pred = model.predict(X_train)
print("Model training complete!")

Model training complete!


In [4]:
# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model

export_model(model, 'output/model/std_lr_model.pkl')

Model successfully exported to output/model/std_lr_model.pkl


## 3. Evaluate Model

In [5]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
extra = ExtraMetrics()
accuracy = accuracy_score(y_test, y_pred)
extra.add(y_test, y_pred, model=model, X=X_test, train_true=y_train, train_pred=y_train_pred, train_time=train_time)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, target_names=['Benign', 'Malignant'], output_dict=True)

print(f"Accuracy: {accuracy:.4f}")
print(f"\nConfusion Matrix:")
print(conf_matrix)
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

Accuracy: 0.9123

Confusion Matrix:
[[72  0]
 [10 32]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.88      1.00      0.94        72
   Malignant       1.00      0.76      0.86        42

    accuracy                           0.91       114
   macro avg       0.94      0.88      0.90       114
weighted avg       0.92      0.91      0.91       114



## 4. Save Results to JSON

In [6]:
# Prepare results dictionary
results = {
    'model_type': 'Standard Logistic Regression',
    'accuracy': float(accuracy),
    'confusion_matrix': conf_matrix.tolist(),
    'classification_report': class_report,
    'parameters': {
        'max_iter': 1000,
        'solver': 'lbfgs',
        'random_state': 42
    }
}
results.update(extra.means())

# Save to JSON
with open('./data/std_lr_report.json', 'w') as f:
    json.dump(results, f, indent=4)

print("Results saved to ./data/std_lr_report.json")

# Also save the flat baseline metrics as CSV, matching the layout used by the
# other BCP model families (positive class = Malignant).
import csv

_pos = class_report['Malignant']
_baseline_row = {
    'model': results['model_type'],
    'accuracy': results['accuracy'],
    'f1_score': _pos['f1-score'],
    'precision': _pos['precision'],
    'recall': _pos['recall'],
    'roc_auc': results['roc_auc'],
    'train_accuracy': results['train_accuracy'],
    'balanced_accuracy': results['balanced_accuracy'],
    'train_time': results['train_time'],
}

with open('./data/baseline_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(_baseline_row))
    writer.writeheader()
    writer.writerow(_baseline_row)

print("Results saved to ./data/baseline_results.csv")

print("Added metrics:")
print(extra.describe())

Results saved to ./data/std_lr_report.json
Added metrics:
  Train Acc:      0.9407 ± 0.0000
  Balanced Acc:   0.8810 ± 0.0000
  AUROC:          0.9980 ± 0.0000
  Train Time (s): 0.0020 ± 0.0000


In [7]:
# Verify saved results
with open('./data/std_lr_report.json', 'r') as f:
    loaded_results = json.load(f)

print("Saved Results Summary:")
print(f"  Model: {loaded_results['model_type']}")
print(f"  Accuracy: {loaded_results['accuracy']:.4f}")
print(f"  Precision (Benign): {loaded_results['classification_report']['Benign']['precision']:.4f}")
print(f"  Precision (Malignant): {loaded_results['classification_report']['Malignant']['precision']:.4f}")
print(f"  Recall (Benign): {loaded_results['classification_report']['Benign']['recall']:.4f}")
print(f"  Recall (Malignant): {loaded_results['classification_report']['Malignant']['recall']:.4f}")

Saved Results Summary:
  Model: Standard Logistic Regression
  Accuracy: 0.9123
  Precision (Benign): 0.8780
  Precision (Malignant): 1.0000
  Recall (Benign): 1.0000
  Recall (Malignant): 0.7619
